# 모델 비교 / 선정 노트북

v3.6 설계서 1~3단계 처리.

- 1단계: Top-3 ≥ 27.7% 임계값 필터 (yolov8n 베이스라인 conf 스캔 best값)
- 2단계: Top-1 정답률 정렬
- 3단계: 상위 3개로 후보 압축 (Top-1 동률 묶음 포함)

입력: `results/models/*.json`

출력: 4단계(재현성 검증) 진입할 후보 모델

## 환경 셋업

In [1]:
from pathlib import Path
import json, os, unicodedata, yaml
from PIL import Image
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r'D:\pill_detection')
TEST_IMG_DIR = PROJECT_ROOT / 'eval' / 'test_images'
LABEL_DIR = PROJECT_ROOT / 'eval' / 'test_labels' / 'test_labels'
MODELS_JSON_DIR = PROJECT_ROOT / 'results' / 'models'
PILL_YAML = PROJECT_ROOT / 'data' / 'pill.yaml'

THRESHOLD_TOP3 = 13/47       # 베이스라인 yolov8n 점수와 동률 이상 (= 27.7%)
TOP_K = 3                    # 상위 3개로 후보 압축
IOU_MATCH = 0.5

def nfc(s):
    return unicodedata.normalize('NFC', s)

# 경로 검증
for name, p in [('TEST_IMG_DIR', TEST_IMG_DIR), ('LABEL_DIR', LABEL_DIR),
                ('MODELS_JSON_DIR', MODELS_JSON_DIR), ('PILL_YAML', PILL_YAML)]:
    print(f"{name}: {'OK' if p.exists() else 'MISSING'} ({p})")


TEST_IMG_DIR: OK (D:\pill_detection\eval\test_images)
LABEL_DIR: OK (D:\pill_detection\eval\test_labels\test_labels)
MODELS_JSON_DIR: OK (D:\pill_detection\results\models)
PILL_YAML: OK (D:\pill_detection\data\pill.yaml)


## GT 47 bbox 빌드

`pill.yaml`의 `names` 리스트 = 학습용 class_idx 매핑 (CSV sorted 아님).
NFC 정규화는 macOS NFD 호환용.

In [2]:
# class_idx → dl_name 매핑
with open(PILL_YAML, 'r', encoding='utf-8') as f:
    yaml_data = yaml.safe_load(f)
class_idx_to_name = {i: nfc(n) for i, n in enumerate(yaml_data['names'])}

# 이미지 stem → path
img_files = {nfc(p.stem): p for p in TEST_IMG_DIR.glob('*')}

# GT 빌드
GT_LABELS = {}
for txt_path in sorted(LABEL_DIR.glob('*.txt')):
    stem = nfc(txt_path.stem)
    if stem not in img_files:
        print(f"⚠️ 매칭 안 됨: {stem}")
        continue
    img_w, img_h = Image.open(img_files[stem]).size
    bboxes = []
    with open(txt_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls = int(parts[0])
            cx, cy, w, h = map(float, parts[1:])
            bboxes.append({
                'bbox': [(cx-w/2)*img_w, (cy-h/2)*img_h, (cx+w/2)*img_w, (cy+h/2)*img_h],
                'dl_name': class_idx_to_name.get(cls, f'unknown_{cls}'),
            })
    GT_LABELS[stem] = {'img_width': img_w, 'img_height': img_h, 'bboxes': bboxes}

total_gt = sum(len(v['bboxes']) for v in GT_LABELS.values())
print(f"GT 이미지: {len(GT_LABELS)}장 / 33")
print(f"GT bbox 합계: {total_gt} / 47")

# 다중 알약 이미지 스폿 체크
multi = [(s, [b['dl_name'] for b in v['bboxes']]) for s, v in GT_LABELS.items() if len(v['bboxes']) > 1]
print(f"\n다중 알약 이미지: {len(multi)}장")
for s, names in multi[:3]:
    print(f"  {s}: {names}")


GT 이미지: 33장 / 33
GT bbox 합계: 47 / 47

다중 알약 이미지: 13장
  놀텍정 10mg: ['놀텍정 10mg', '놀텍정 10mg']
  디카테오정: ['디카테오정', '디카테오정', '디카테오정']
  라시도필캡슐(비매품): ['라시도필캡슐(비매품)', '라시도필캡슐(비매품)']


## 채점 헬퍼

GT bbox별로 IoU 가장 높은 prediction을 greedy 매칭.
매칭된 prediction의 `top3_class_name`으로 Top-1 / Top-3 판정.
이미지 단위 = 그 이미지의 모든 GT bbox가 정답 매칭되어야 카운트.

In [3]:
def iou_xyxy(b1, b2):
    x1, y1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    x2, y2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    a1 = (b1[2] - b1[0]) * (b1[3] - b1[1])
    a2 = (b2[2] - b2[0]) * (b2[3] - b2[1])
    return inter / (a1 + a2 - inter + 1e-9)


def evaluate_model(json_path, gt_labels, iou_thr=IOU_MATCH):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    n_top1 = n_top3 = n_match = 0
    img_top1_full = img_top3_full = 0
    n_false_pos = 0
    total_gt = sum(len(v['bboxes']) for v in gt_labels.values())

    res_by_stem = {}
    for r in data.get('results', []):
        stem = nfc(os.path.splitext(r['filename'])[0])
        res_by_stem[stem] = r

    for stem, info in gt_labels.items():
        gt_bboxes = info['bboxes']
        if stem not in res_by_stem:
            # JSON에 해당 이미지 결과 없음 → 이미지 단위 둘 다 실패
            continue
        preds = res_by_stem[stem].get('predictions', [])
        used = set()
        img_all_top1 = bool(gt_bboxes)
        img_all_top3 = bool(gt_bboxes)

        for gt in gt_bboxes:
            best_iou, best_idx = 0.0, -1
            for j, p in enumerate(preds):
                if j in used:
                    continue
                iou = iou_xyxy(gt['bbox'], p['bbox'])
                if iou > best_iou:
                    best_iou, best_idx = iou, j

            if best_iou >= iou_thr and best_idx >= 0:
                n_match += 1
                used.add(best_idx)
                top3 = [nfc(n) for n in preds[best_idx].get('top3_class_name', [])]
                gt_name = nfc(gt['dl_name'])
                if top3 and top3[0] == gt_name:
                    n_top1 += 1
                else:
                    img_all_top1 = False
                if gt_name in top3:
                    n_top3 += 1
                else:
                    img_all_top3 = False
            else:
                img_all_top1 = False
                img_all_top3 = False

        if img_all_top1:
            img_top1_full += 1
        if img_all_top3:
            img_top3_full += 1
        n_false_pos += len(preds) - len(used)

    return {
        'exp_name': data.get('exp_name', os.path.basename(json_path)),
        'author': data.get('author', '?'),
        'framework': data.get('model_framework', '?'),
        'params_M': data.get('model_params_M', None),
        'conf_threshold': data.get('conf_threshold', None),
        'ms_per_image': data.get('ms_per_image') or data.get('metrics', {}).get('ms_per_image'),
        'detection_rate': n_match / total_gt if total_gt else 0,
        'top1_accuracy': n_top1 / total_gt if total_gt else 0,
        'top3_accuracy': n_top3 / total_gt if total_gt else 0,
        'image_top1': img_top1_full / 33,
        'image_top3': img_top3_full / 33,
        'n_top1': n_top1,
        'n_top3': n_top3,
        'n_match': n_match,
        'n_false_pos': n_false_pos,
        'json_path': str(json_path),
    }


## 결과 JSON 일괄 로드 + 채점

`results/models/` 안의 모든 `.json` 자동 스캔.

In [4]:
json_paths = sorted(MODELS_JSON_DIR.glob('*.json'))
print(f"발견된 JSON: {len(json_paths)}개")
for p in json_paths:
    print(f"  - {p.name}")

print()
records = []
for p in json_paths:
    try:
        rec = evaluate_model(p, GT_LABELS)
        records.append(rec)
        print(f"✅ {rec['exp_name']}: Top-1={rec['top1_accuracy']:.1%}, Top-3={rec['top3_accuracy']:.1%}")
    except Exception as e:
        print(f"❌ {p.name}: {e}")

df = pd.DataFrame(records)
print(f"\n채점 완료: {len(df)}개 모델")


발견된 JSON: 8개
  - 06_Pill_FasterRCNN_MobileNetV3LargeFPN_baseline_seed42_submission.json
  - 06_Pill_YOLO11s_baseline_seed42_submission.json
  - 06_Pill_YOLOv8s_baseline_seed42_submission.json
  - 99_Pill_YOLOv8n_baseline_seed42_submission.json
  - rtdetr_synth_result.json
  - rtmdet_tiny_synth_e15_results.json
  - yolo11n_synth_result.json
  - yolox_s_top3_result.json

✅ 06_Pill_FasterRCNN_MobileNetV3LargeFPN_baseline_seed42: Top-1=6.4%, Top-3=14.9%
✅ 06_Pill_YOLO11s_baseline_seed42: Top-1=17.0%, Top-3=29.8%
✅ 06_Pill_YOLOv8s_baseline_seed42: Top-1=21.3%, Top-3=29.8%
✅ 99_Pill_YOLOv8n_baseline_seed42: Top-1=14.9%, Top-3=27.7%
✅ rtdetr_synth: Top-1=21.3%, Top-3=27.7%
✅ rtmdet_tiny_synth_e15: Top-1=34.0%, Top-3=46.8%
✅ yolo11n_synth: Top-1=14.9%, Top-3=27.7%
✅ yolox_s_synth_augFalse_seed42_top3: Top-1=21.3%, Top-3=23.4%

채점 완료: 8개 모델


## 1단계: 임계값 필터

Top-3 ≥ 25.5% (귀한님 yolov8n + 합성 베이스라인 동률 이상).

In [5]:
df_sorted = df.sort_values('top3_accuracy', ascending=False).reset_index(drop=True)
df_sorted['임계값_통과'] = df_sorted['top3_accuracy'] >= THRESHOLD_TOP3
df_sorted['통과'] = df_sorted['임계값_통과'].apply(lambda x: '✅' if x else '❌')

cols_show = ['통과', 'exp_name', 'author', 'top3_accuracy', 'top1_accuracy',
             'detection_rate', 'ms_per_image', 'params_M']
display1 = df_sorted[cols_show].copy()
for c in ['top3_accuracy', 'top1_accuracy', 'detection_rate']:
    display1[c] = display1[c].apply(lambda x: f'{x:.1%}')

print(f"임계값: Top-3 ≥ {THRESHOLD_TOP3:.1%}\n")
print(display1.to_string(index=False))

passed_models = df_sorted[df_sorted['임계값_통과']].reset_index(drop=True)
print(f"\n통과 모델: {len(passed_models)} / {len(df_sorted)}")


임계값: Top-3 ≥ 27.7%

통과                                               exp_name author top3_accuracy top1_accuracy detection_rate  ms_per_image  params_M
 ✅                                  rtmdet_tiny_synth_e15     wony98         46.8%         34.0%          97.9%       120.540      4.91
 ✅                        06_Pill_YOLO11s_baseline_seed42    팀원 A         29.8%         17.0%          83.0%        67.575      9.47
 ✅                        06_Pill_YOLOv8s_baseline_seed42    팀원 A         29.8%         21.3%          83.0%        64.084     11.18
 ✅                        99_Pill_YOLOv8n_baseline_seed42    팀원 A         27.7%         14.9%          74.5%       123.668      3.35
 ✅                                           rtdetr_synth     팀원 D         27.7%         21.3%          91.5%        68.100     33.00
 ✅                                          yolo11n_synth     팀원 D         27.7%         14.9%          91.5%        25.700      2.60
 ❌                     yolox_s_synth_augFalse

## 2단계: Top-1 정답률 정렬

통과 모델만 Top-1 내림차순. HealthEat "1개 정답" 컨셉 직결.

In [6]:
if len(passed_models) == 0:
    print("⚠️ 임계값 통과 모델 없음. 다음 단계 진행 불가.")
else:
    top1_sorted = passed_models.sort_values('top1_accuracy', ascending=False).reset_index(drop=True)
    cols2 = ['exp_name', 'author', 'top1_accuracy', 'image_top1', 'top3_accuracy',
             'detection_rate', 'ms_per_image', 'params_M', 'n_top1']
    display2 = top1_sorted[cols2].copy()
    for c in ['top1_accuracy', 'image_top1', 'top3_accuracy', 'detection_rate']:
        display2[c] = display2[c].apply(lambda x: f'{x:.1%}')
    print("Top-1 정답률 내림차순:\n")
    print(display2.to_string(index=False))


Top-1 정답률 내림차순:

                       exp_name author top1_accuracy image_top1 top3_accuracy detection_rate  ms_per_image  params_M  n_top1
          rtmdet_tiny_synth_e15     wony98         34.0%      30.3%         46.8%          97.9%       120.540      4.91      16
06_Pill_YOLOv8s_baseline_seed42    팀원 A         21.3%      15.2%         29.8%          83.0%        64.084     11.18      10
                   rtdetr_synth     팀원 D         21.3%      24.2%         27.7%          91.5%        68.100     33.00      10
06_Pill_YOLO11s_baseline_seed42    팀원 A         17.0%       9.1%         29.8%          83.0%        67.575      9.47       8
99_Pill_YOLOv8n_baseline_seed42    팀원 A         14.9%       9.1%         27.7%          74.5%       123.668      3.35       7
                  yolo11n_synth     팀원 D         14.9%      12.1%         27.7%          91.5%        25.700      2.60       7


## 3단계: 상위 3개로 후보 압축

Top-1 정답률 기준 상위 3개로 좁힘.
3등과 동률(같은 n_top1) 모델은 모두 후보에 포함 → 4단계 재현성 검증으로 결판.

In [7]:
if len(passed_models) == 0:
    print("⚠️ 임계값 통과 모델 없음. 다음 단계 진행 불가.")
    candidates = passed_models
else:
    top1_sorted = passed_models.sort_values('top1_accuracy', ascending=False).reset_index(drop=True)

    if len(top1_sorted) <= TOP_K:
        candidates = top1_sorted.copy()
    else:
        # 상위 TOP_K + 3등 n_top1과 동률인 모델 모두 포함
        cutoff_n_top1 = top1_sorted.iloc[TOP_K - 1]['n_top1']
        candidates = top1_sorted[top1_sorted['n_top1'] >= cutoff_n_top1].reset_index(drop=True)

    cols3 = ['exp_name', 'author', 'n_top1', 'top1_accuracy', 'image_top1',
             'top3_accuracy', 'detection_rate', 'ms_per_image', 'params_M']
    display3 = candidates[cols3].copy()
    for c in ['top1_accuracy', 'image_top1', 'top3_accuracy', 'detection_rate']:
        display3[c] = display3[c].apply(lambda x: f'{x:.1%}')

    print(f"상위 {TOP_K}개 + 동률 묶음:\n")
    print(display3.to_string(index=False))
    print(f"\n🥇 4단계 재현성 검증 후보: {len(candidates)}개")
    for i, row in candidates.iterrows():
        print(f"   {i+1}. {row['exp_name']} ({row['author']}) — Top-1 {row['top1_accuracy']:.1%}")

상위 3개 + 동률 묶음:

                       exp_name author  n_top1 top1_accuracy image_top1 top3_accuracy detection_rate  ms_per_image  params_M
          rtmdet_tiny_synth_e15     wony98      16         34.0%      30.3%         46.8%          97.9%       120.540      4.91
06_Pill_YOLOv8s_baseline_seed42    팀원 A      10         21.3%      15.2%         29.8%          83.0%        64.084     11.18
                   rtdetr_synth     팀원 D      10         21.3%      24.2%         27.7%          91.5%        68.100     33.00

🥇 4단계 재현성 검증 후보: 3개
   1. rtmdet_tiny_synth_e15 (wony98) — Top-1 34.0%
   2. 06_Pill_YOLOv8s_baseline_seed42 (팀원 A) — Top-1 21.3%
   3. rtdetr_synth (팀원 D) — Top-1 21.3%


In [8]:
import json, unicodedata
def nfc(s): return unicodedata.normalize('NFC', s.strip()) if s else s
def iou(a,b):
    x1,y1=max(a[0],b[0]),max(a[1],b[1]); x2,y2=min(a[2],b[2]),min(a[3],b[3])
    inter=max(0,x2-x1)*max(0,y2-y1)
    aa=(a[2]-a[0])*(a[3]-a[1]); ab=(b[2]-b[0])*(b[3]-b[1])
    return inter/(aa+ab-inter+1e-9)

p = r'D:\pill_detection\results\models\rtmdet_tiny_synth_e15_results.json'
with open(p, encoding='utf-8') as f:
    d = json.load(f)

from collections import Counter
matched_top1_correct = []   # 정답 케이스 GT
matched_top1_wrong = []     # 오답 케이스 (GT, pred top1)

for r in d['results']:
    for gt in r['gt_bboxes']:
        gt_name = nfc(gt['dl_name'])
        best_iou, best_pred = 0, None
        for pred in r['predictions']:
            i = iou(gt['bbox'], pred['bbox'])
            if i >= 0.5 and i > best_iou:
                best_iou, best_pred = i, pred
        if best_pred is None: continue
        top1 = nfc(best_pred['top3_class_name'][0])
        if top1 == gt_name:
            matched_top1_correct.append(gt_name)
        else:
            matched_top1_wrong.append((gt_name, top1))

print(f'매칭 박스 {len(matched_top1_correct)+len(matched_top1_wrong)}개')
print(f'\nTop-1 정답 ({len(matched_top1_correct)}개):')
for cls, cnt in Counter(matched_top1_correct).most_common():
    print(f'  {cnt}회: {cls}')
print(f'\nTop-1 오답 ({len(matched_top1_wrong)}개) [GT → Pred]:')
for (gt, pred), cnt in Counter(matched_top1_wrong).most_common():
    print(f'  {cnt}회: {gt} → {pred}')

매칭 박스 46개

Top-1 정답 (16개):
  2회: 놀텍정 10mg
  2회: 라시도필캡슐(비매품)
  1회: 게보린정 300mg/PTP
  1회: 노바스크정 5mg
  1회: 뮤테란캡슐 100mg
  1회: 비모보정 500/20mg
  1회: 비오플250캡슐 282.5mg
  1회: 아보다트연질캡슐 0.5mg
  1회: 엔테론정 150mg
  1회: 인사돌플러스정 35mg/PTP
  1회: 졸로푸트정 100mg
  1회: 종근당글리아티린연질캡슐(콜린알포세레이트)
  1회: 지르텍정(세티리진염산염)
  1회: 트루비타정 60mg/병

Top-1 오답 (30개) [GT → Pred]:
  2회: 디카테오정 → 디카테오엘씨정 10mg/병
  2회: 아토르바정 10mg → 토비애즈서방정 4mg
  1회: 글리아타민연질캡슐 → 놀텍정 10mg
  1회: 낙소졸정 500/20mg → 엘스테인정(에르도스테인)
  1회: 넥시움정 40mg → 제미메트서방정 50/1000mg
  1회: 디카테오정 → 씨즈날정(세티리진염산염)
  1회: 마이칼디정 → 레일라정
  1회: 마이칼디정 → 삐콤씨에프정 618.6mg/병
  1회: 맥시부펜이알정 300mg → 노바스크정 5mg
  1회: 맥시부펜이알정 300mg → 씨즈날정(세티리진염산염)
  1회: 비타비백정 100mg/병 → 넥시움정 40mg
  1회: 아스피린프로텍트정 100mg → 에어탈정(아세클로페낙)
  1회: 엔테론정 150mg → 글리틴정(콜린알포세레이트)
  1회: 울트라셋이알서방정 → 글리틴정(콜린알포세레이트)
  1회: 인사돌플러스정 35mg/PTP → 큐시드정 31.5mg/PTP
  1회: 자누비아정 50mg → 제라타딘정(로라타딘)
  1회: 자누비아정 50mg → 쎄로켈정 100mg
  1회: 자트랄엑스엘정 10mg → 베시케어정 10mg
  1회: 자트랄엑스엘정 10mg → 디카맥스디플러스정
  1회: 졸로푸트정 100mg → 아토젯정 10/40mg
  1회: 종근당글리아티린연질캡슐(콜린알포세레이트

In [9]:
# Jupyter / Python 셀에서 실행
import pandas as pd

df = pd.read_pickle(r"D:\pill_detection\eda\merged_dataset.pkl")
print(f"전체: {len(df):,}행 × {len(df.columns)}컬럼")

# 필요 컬럼만 추출 + dl_name별 unique
meta = (df.groupby('img_dl_name')
        .agg({'img_drug_shape': 'first',
              'img_color_class1': 'first'})
        .reset_index())

# 컬럼명 정리 (5단계 b 노트북이 dl_name, drug_shape, color_class1 기대)
meta = meta.rename(columns={
    'img_dl_name': 'dl_name',
    'img_drug_shape': 'drug_shape',
    'img_color_class1': 'color_class1'
})

# 저장
out_path = r'D:\pill_detection\eda\merged_dataset_meta.csv'
meta.to_csv(out_path, index=False, encoding='utf-8-sig')

print(f"\n저장: {out_path}")
print(f"고유 알약: {len(meta)}종")
print(f"\n샘플:")
print(meta.head(10).to_string(index=False))
print(f"\n모양 분포:")
print(meta['drug_shape'].value_counts())
print(f"\n색상 분포:")
print(meta['color_class1'].value_counts().head(10))

전체: 41,225행 × 76컬럼

저장: D:\pill_detection\eda\merged_dataset_meta.csv
고유 알약: 118종

샘플:
           dl_name drug_shape color_class1
       가바토파정 100mg       None         None
    게보린정 300mg/PTP        삼각형           분홍
         글리아타민연질캡슐        타원형           갈색
    글리틴정(콜린알포세레이트)        타원형           주황
기넥신에프정(은행엽엑스)(수출용)        장방형           초록
     낙소졸정 500/20mg        타원형           노랑
         넥시움정 40mg        장방형           분홍
         노바스크정 5mg        팔각형           하양
          놀텍정 10mg        타원형           주황
      뉴로메드정(옥시라세탐)        장방형           하양

모양 분포:
drug_shape
원형     40
장방형    37
타원형    35
기타      2
삼각형     1
팔각형     1
육각형     1
Name: count, dtype: int64

색상 분포:
color_class1
하양        47
노랑        18
분홍        17
주황         9
갈색         8
초록         6
파랑         5
빨강         3
연두         2
노랑, 투명     1
Name: count, dtype: int64


## 다음 단계

이 노트북 3단계 결과 = 4단계 재현성 검증(seed=100 재학습) 진입할 후보 모델.

다음 노트북: `model_comparison.ipynb` (4~5단계)